# YOLOv8 Eğitimi — Google Colab

Bu notebook **`04_prepare_yolo.py`** ve **`05_train_yolo.py`** adımlarını
Google Colab üzerinde çalıştırır. Dataset Google Drive'dan okunur;
eğitim sonuçları Drive'a geri kaydedilir.

## Adımlar
| # | Script | Ne yapar |
|---|---|---|
| 0–5 | *(kurulum)* | GPU, Drive, paketler, kod kopyalama, yol eşleme |
| **6** | `04_prepare_yolo.py` | DICOM → PNG + YOLO etiket + dataset.yaml |
| **7** | `05_train_yolo.py` | YOLOv8 model eğitimi |
| 8–10 | *(sonuç)* | Drive kaydetme, grafikler, inference |

## Hızlı Başlangıç
1. **Runtime → Change runtime type → T4 GPU** seçin.
2. Yalnızca **`[1] Kullanıcı Ayarları`** hücresini düzenleyin.
3. **Runtime → Run all** ile çalıştırın.

## Google Drive Beklenen Klasör Yapısı
```
MyDrive/
└── abdomen/                        ← DRIVE_ABDOMEN
    ├── Bilgi.xlsx
    ├── Egitim Verisi/              ← DICOM dosyaları
    ├── Test Verisi/
    └── abdomen_project/
        ├── src/
        ├── scripts/
        └── outputs/
            └── splits/             ← manifest.csv + fold*.csv (01_split.py çıktısı)
```


In [ ]:
# ─── 0. GPU Kontrolü & Google Drive Bağlama ───────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory // (1024**3)
    print(f"GPU : {name}  ({mem} GB)")
else:
    print("GPU bulunamadi! Runtime > Change runtime type > T4 GPU secin.")


## [1] Kullanıcı Ayarları — yalnızca bu hücreyi düzenleyin


In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

DATA_ROOT = Path(os.environ.get("TR_ABDOMEN_BASE", r"D:/makale-pdf/Proje/abdomen"))
PROJE     = Path(os.environ.get("TR_ABDOMEN_PROJE", r"D:/makale-pdf/Proje"))
CODE      = Path(os.environ.get("TR_ABDOMEN_CODE",  r"D:/makale-pdf/Proje/code"))
EGITIM_DIR = Path(os.environ.get("ABDOMEN_TRAIN_DIR", DATA_ROOT / "Egitim Verisi"))
YARISMA_DIR = Path(os.environ.get("ABDOMEN_TEST_DIR", DATA_ROOT / "Test Verisi"))
OUT_DIR = Path(os.environ.get("ABDOMEN_OUT_DIR",  r"D:/makale-pdf/Proje/outputs"))


DET_DATA_DIR = Path(os.environ.get("ABDOMEN_DET_DATA_DIR", OUT_DIR / "det_data"))
DET_DATA_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(CODE))


In [ ]:
from pathlib import Path
    # ══════════════════════════════════════════════════════════════════════════════
#  BURASI DEĞİŞTİRİLECEK TEK HÜCRE
# ══════════════════════════════════════════════════════════════════════════════
# Splits (01_split.py çıktısı): manifest.csv + fold0_train.csv vb.
DRIVE_SPLITS = OUT_DIR / "splits"
# Eğitim DICOMlarının kök klasörü
# Eğitilecek fold (0 – 4)
FOLD = 0
# YOLO PNG çıktıları nerede oluşturulsun?
#   True  → /content/det_data  (hızlı; session bitince silinir)
#   False → PROJE / outputs / det_data  (kalıcı ama yavaş)
LOCAL_DET_DATA = True
# Eğitim run klasörünün Drive'daki hedefi
DRIVE_RESULTS = OUT_DIR / "yolo_results"

# ══════════════════════════════════════════════════════════════════════════════

print(f"Drive abdomen  : {DATA_ROOT}")
print(f"Drive splits   : {DRIVE_SPLITS}")
print(f"Train DICOM    : {EGITIM_DIR}")
print(f"Fold           : {FOLD}")
print(f"Local det_data : {LOCAL_DET_DATA}")
print()
for p in [DATA_ROOT, PROJE, DRIVE_SPLITS, EGITIM_DIR]:
    status = "OK" if p.exists() else "BULUNAMADI"
    print(f"  [{status}] {p}")


## [2] Paket Kurulumu


In [ ]:
%%capture install_log
!pip install -q ultralytics pydicom opencv-python-headless tqdm pandas openpyxl scikit-learn pyyaml


In [ ]:
import ultralytics, pydicom, cv2
print(f"ultralytics : {ultralytics.__version__}")
print(f"pydicom     : {pydicom.__version__}")
print(f"OpenCV      : {cv2.__version__}")


## [3] Proje Kodunu Colab Diskine Kopyala

Drive I/O gecikme olmadan çalışması için `abdomen_project/` Colab yerel diskine kopyalanır.


In [ ]:
import shutil, sys
from pathlib import Path

COLAB_PROJECT = Path("/content/abdomen_project")

if COLAB_PROJECT.exists():
    shutil.rmtree(COLAB_PROJECT)
shutil.copytree(str(PROJE), str(COLAB_PROJECT))

if str(COLAB_PROJECT) not in sys.path:
    sys.path.insert(0, str(COLAB_PROJECT))

print(f"Proje kopyalandi : {COLAB_PROJECT}")
print(f"sys.path[0]      : {sys.path[0]}")


## [4] Ortam Değişkenlerini Ayarla

`export_yolo_dataset` içindeki `ProcessPoolExecutor` alt süreçleri bu değişkenleri miras alır;  
dolayısıyla `config.py` her alt süreçte de doğru yolları okur.


In [ ]:
import os
from pathlib import Path



# os.environ["ABDOMEN_PROJECT_ROOT"] = str(COLAB_PROJECT)
# os.environ["ABDOMEN_DATA_ROOT"]    = str(DRIVE_ABDOMEN)
# os.environ["ABDOMEN_TRAIN_DIR"]    = str(DRIVE_TRAIN_DIR)
# os.environ["ABDOMEN_TEST_DIR"]     = str(DRIVE_ABDOMEN / "Test Verisi")
# os.environ["ABDOMEN_BILGI_XLSX"]   = str(DRIVE_ABDOMEN / "Bilgi.xlsx")
# os.environ["ABDOMEN_SPLIT_DIR"]    = str(DRIVE_SPLITS)   # manifest remap'ten sonra güncellenecek
# os.environ["ABDOMEN_DET_DATA_DIR"] = str(DET_DATA_DIR)
# os.environ["ABDOMEN_OUT_DIR"]      = str(COLAB_PROJECT / "outputs")

for k, v in sorted(os.environ.items()):
    if k.startswith("ABDOMEN_"):
        print(f"{k:30s} = {v}")


## [5] Manifest DICOM Yollarını Yeniden Eşle

`manifest.csv` yerel Mac'te oluşturulmuşsa içindeki DICOM yolları  
Colab'da geçersizdir. Bu hücre yolları Drive eşdeğerine çevirir.


In [ ]:
import shutil
import pandas as pd
from pathlib import Path

manifest_src = DRIVE_SPLITS / "manifest.csv"
assert manifest_src.exists(), f"manifest.csv bulunamadi: {manifest_src}"

df = pd.read_csv(manifest_src)
sample = str(df["dicom_path"].iloc[0])
print(f"Orijinal yol ornek : {sample}")

COLAB_SPLITS = Path("/content/splits_colab")
COLAB_SPLITS.mkdir(exist_ok=True)

already_drive = sample.startswith("/content/drive") or sample.startswith(str(DATA_ROOT))

if already_drive:
    print("Yollar zaten Drive formatinda, esleme atlandi.")
else:
    # Ortak anchor ('abdomen') uzerinden Drive yoluna cevir
    ANCHOR = "abdomen"
    drive_parent = str(DATA_ROOT.parent)  # .../MyDrive

    def _remap(p):
        p = str(p)
        if ANCHOR in p:
            idx = p.index(ANCHOR)
            return drive_parent + "/" + p[idx:]
        return p

    df["dicom_path"] = df["dicom_path"].apply(_remap)
    print(f"Yeniden eslendi    : {df['dicom_path'].iloc[0]}")

# Colab yerel diskine kaydet
df.to_csv(COLAB_SPLITS / "manifest.csv", index=False)
for f_csv in DRIVE_SPLITS.glob("fold*.csv"):
    shutil.copy(f_csv, COLAB_SPLITS / f_csv.name)

# SPLIT_DIR'i güncellenmiş manifest'e yönlendir
os.environ["ABDOMEN_SPLIT_DIR"] = str(COLAB_SPLITS)

print(f"\nCOLAB_SPLITS   : {COLAB_SPLITS}")
print(f"Toplam satir   : {len(df)}")
print(f"Annotasyonlu   : {df['bboxes'].notna().sum()}")

for s in ("train", "val"):
    f = COLAB_SPLITS / f"fold{FOLD}_{s}.csv"
    print(f"  fold{FOLD}_{s}.csv : {'OK' if f.exists() else 'EKSIK'}")


## [6] 04_prepare_yolo.py — YOLO Dataset Hazırla

```python
# scripts/04_prepare_yolo.py özeti
from src.detection import export_yolo_dataset
path = export_yolo_dataset(fold=FOLD)
print('Hazır:', path)
```

`export_yolo_dataset` fonksiyonu:
- DICOM dosyalarını 3 kanallı PNG'ye dönüştürür (soft_tissue / liver / calcified)
- Her kesit için YOLO txt etiket dosyası oluşturur (6 üst sınıf)
- `dataset.yaml` yazar

> **Süre:** Dataset büyüklüğüne göre 10–40 dakika.  
> `LOCAL_DET_DATA=True` ile `/content/det_data`'ya yazıldığında Drive'a göre çok daha hızlıdır.


In [ ]:
# ── 04_prepare_yolo.py ───────────────────────────────────────────────────
# Ortam değişkenleri önceden ayarlandığı için import sonrasında yollar doğru okunur
from src.detection import export_yolo_dataset

print(f"Cikti dizini : {DET_DATA_DIR}")
print(f"Fold         : {FOLD}")
print("Export basliyor...")

fold_dir = export_yolo_dataset(fold=FOLD, out_root=DET_DATA_DIR)
print(f"\nHazir: {fold_dir}")

for split in ("train", "val"):
    imgs   = list((fold_dir / "images" / split).glob("*.png"))
    labels = list((fold_dir / "labels" / split).glob("*.txt"))
    pos    = sum(1 for f in labels if f.read_text().strip())
    print(f"  {split:5s}: {len(imgs):5d} goruntu | {pos:5d} pozitif etiket")

print("\ndataset.yaml:")
print((fold_dir / "dataset.yaml").read_text())


## [7] 05_train_yolo.py — YOLOv8 Eğitimi

```python
# scripts/05_train_yolo.py özeti
from src.detection import train_yolo
w = train_yolo(fold=FOLD)
print('YOLO best weights:', w)
```

`train_yolo()` içinde:
- `dataset.yaml` yoksa `export_yolo_dataset()` otomatik çağrılır
- MPS → CUDA → CPU öncelik sırasıyla cihaz seçilir (Colab'da CUDA:0)
- `amp=True` (FP16), `cache="disk"`, `patience=20`

| Hiperparametre | Varsayılan |
|:---|:---|
| Model | yolov8x.pt (ImageNet pretrained) |
| imgsz | 768 |
| epochs | 50 |
| batch | 16 |
| mosaic | 0.5 |
| mixup | 0.15 |

Değiştirmek için aşağıdaki `TRAIN_KWARGS` sözlüğünü düzenleyin.


In [ ]:
# ── 05_train_yolo.py ─────────────────────────────────────────────────────
from src.detection import train_yolo
from src.config import DEFAULT_DET
from pathlib import Path

# train_yolo() scriptteki çağrının birebir karşılığı:
#   w = train_yolo(FOLD)
# Aşağıda Colab için workers/device sabitlenmiş, diğerleri DEFAULT_DET'ten gelir.

import src.detection as _det
_det.DET_DATA_DIR = DET_DATA_DIR   # export adımında kullandığımız dizinle aynı

# Colab için workers ve project yolunu override eden ince sarmalayıcı
import os, torch
from ultralytics import YOLO

cfg      = DEFAULT_DET
project  = "/content/runs/det"
run_name = f"fold{FOLD}_{Path(cfg.model).stem}"
yaml     = DET_DATA_DIR / f"fold{FOLD}" / "dataset.yaml"

# Özelleştirilebilir — değiştirmek istemiyorsanız bu bloğu atlayın
TRAIN_KWARGS = dict(
    data          = str(yaml),
    imgsz         = cfg.img_size,
    epochs        = cfg.epochs,
    batch         = cfg.batch_size,
    mosaic        = cfg.mosaic,
    mixup         = cfg.mixup,
    project       = project,
    name          = run_name,
    seed          = 42,
    deterministic = False,
    close_mosaic  = 10,
    patience      = 20,
    device        = "0",   # Colab T4
    workers       = 2,     # Colab çekirdek sayısı
    cache         = "disk",
    amp           = True,
)

from src.config import SUPER_CLASSES
print(f"Model    : {cfg.model}")
print(f"Epochs   : {cfg.epochs}")
print(f"ImgSize  : {cfg.img_size}")
print(f"Batch    : {cfg.batch_size}")
print(f"Run dir  : {project}/{run_name}")
print(f"Siniflar : {SUPER_CLASSES}")
print("-" * 50)

model   = YOLO(cfg.model)
results = model.train(**TRAIN_KWARGS)

best_weights = Path(project) / run_name / "weights" / "best.pt"
print(f"\nYOLO best weights: {best_weights}")
print(f"Boyut            : {best_weights.stat().st_size / 1e6:.1f} MB")


## [8] Sonuçları Google Drive'a Kaydet


In [ ]:
import shutil
from pathlib import Path
import pandas as pd
from src.config import DEFAULT_DET

src_run = Path("/content/runs/det") / f"fold{FOLD}_{Path(DEFAULT_DET.model).stem}"
dst_run = DRIVE_RESULTS / f"fold{FOLD}"

DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
if dst_run.exists():
    shutil.rmtree(dst_run)
shutil.copytree(str(src_run), str(dst_run))

best = dst_run / "weights" / "best.pt"
last = dst_run / "weights" / "last.pt"
print(f"Kaydedildi  : {dst_run}")
print(f"best.pt     : {best.stat().st_size / 1e6:.1f} MB")
print(f"last.pt     : {last.stat().st_size / 1e6:.1f} MB")

results_csv = dst_run / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    row = df.iloc[-1]
    print("\nSon epoch metrikleri:")
    for col in ["metrics/precision(B)", "metrics/recall(B)",
                "metrics/mAP50(B)", "metrics/mAP50-95(B)"]:
        if col in row:
            print(f"  {col:30s}: {row[col]:.4f}")


## [9] Eğitim Grafiklerini Görselleştir


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from src.config import DEFAULT_DET

run_dir = Path("/content/runs/det") / f"fold{FOLD}_{Path(DEFAULT_DET.model).stem}"

def _show(path, title="", figsize=(16, 6)):
    if not path.exists():
        print(f"Bulunamadi: {path}")
        return
    img = mpimg.imread(str(path))
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.axis("off")
    if title:
        plt.title(title, fontsize=13)
    plt.tight_layout()
    plt.show()

_show(run_dir / "results.png", "Egitim Metrikleri", figsize=(18, 8))

for cm in run_dir.glob("confusion_matrix*.png"):
    _show(cm, cm.stem, figsize=(10, 9))

for vp in sorted(run_dir.glob("val_batch*_pred.jpg"))[:2]:
    _show(vp, vp.stem, figsize=(14, 7))


## [10] (Opsiyonel) Tek Görüntü Üzerinde Inference


In [ ]:
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from src.config import DEFAULT_DET, SUPER_CLASSES

best_weights = Path("/content/runs/det") / \
    f"fold{FOLD}_{Path(DEFAULT_DET.model).stem}" / "weights" / "best.pt"

val_imgs = sorted((DET_DATA_DIR / f"fold{FOLD}" / "images" / "val").glob("*.png"))
if not val_imgs:
    print("Val PNG bulunamadi.")
else:
    test_img = val_imgs[len(val_imgs) // 2]
    print(f"Test goruntu: {test_img.name}")

    model = YOLO(str(best_weights))
    res   = model.predict(str(test_img), conf=0.25, verbose=False)[0]

    ann = res.plot(line_width=2)
    plt.figure(figsize=(8, 8))
    plt.imshow(ann[..., ::-1])   # BGR → RGB
    plt.axis("off")
    plt.title(f"{test_img.stem}  |  {len(res.boxes)} tespit", fontsize=12)
    plt.tight_layout()
    plt.show()

    for box in res.boxes:
        cls_id = int(box.cls)
        score  = float(box.conf)
        print(f"  Sinif: {SUPER_CLASSES[cls_id]:35s}  Guven: {score:.3f}")
